# 🛍️ Multimodal RAG for E-Commerce Product Assistant
### Using: ChatGroq (LLM) + HuggingFace (Embeddings) + FAISS (Vector DB) + LangChain

**Flow:**
```
User uploads product image
    → Vision LLM generates text description
    → HuggingFace Embeddings encode description
    → FAISS vector search finds similar products
    → ChatGroq generates final answer with context
    → Response shown to user
```

## Cell 1: Install Dependencies

In [1]:
# Install all required packages
!pip install langchain langchain-community langchain-groq
!pip install faiss-cpu sentence-transformers
!pip install Pillow requests python-dotenv
!pip install groq google-generativeai  # Gemini for vision
!pip install datasets  # to load a sample product dataset from HuggingFace Hub

print("✅ All packages installed!")

ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


✅ All packages installed!


ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


## Cell 2: API Keys & Config

In [6]:
# Force remove any existing env variable and reload from .env
import os

# Remove the old key from current environment
if "GROQ_API_KEY" in os.environ:
    del os.environ["GROQ_API_KEY"]
    print("🗑️ Old key removed from environment")

# Now reload from .env file
from dotenv import load_dotenv
load_dotenv(override=True)  # override=True forces .env values to overwrite existing ones

# Verify
key = os.getenv("GROQ_API_KEY")
print(f"✅ New key loaded: {key[:8]}...{key[-4:]}")

🗑️ Old key removed from environment
✅ New key loaded: gsk_Op8Z...3ezS


In [7]:
import os
from dotenv import load_dotenv

load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# Validate keys are loaded
if not GROQ_API_KEY:
    raise ValueError("❌ GROQ_API_KEY not found! Make sure your .env file exists and has the key.")

# Model config
GROQ_MODEL      = "llama-3.3-70b-versatile"                    # text LLM (chat + reasoning)
VISION_MODEL    = "meta-llama/llama-4-scout-17b-16e-instruct"  # vision LLM (image understanding)
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"     # HuggingFace local embeddings
TOP_K_RESULTS   = 5

print("✅ Config loaded!")
print(f"   GROQ_API_KEY : {GROQ_API_KEY[:8]}...{GROQ_API_KEY[-4:]}")
print(f"   Chat LLM     : {GROQ_MODEL}")
print(f"   Vision LLM   : {VISION_MODEL}")
print(f"   Embeddings   : {EMBEDDING_MODEL}")

✅ Config loaded!
   GROQ_API_KEY : gsk_Op8Z...3ezS
   Chat LLM     : llama-3.3-70b-versatile
   Vision LLM   : meta-llama/llama-4-scout-17b-16e-instruct
   Embeddings   : sentence-transformers/all-MiniLM-L6-v2


## Cell 3: Build Sample Product Catalog
> In production you'd load from a real DB. Here we create a small catalog.

In [8]:
import json

# ──────────────────────────────────────────────────────────────────────────────
# Sample product catalog — each product has rich text for embedding
# 💡 TIP for internship: Add image_url field + embed both image & text features
#    (cross-modal fusion = huge differentiator on your resume!)
# ──────────────────────────────────────────────────────────────────────────────
PRODUCT_CATALOG = [
    {
        "id": "P001",
        "name": "Nike Air Max 270 White Sneakers",
        "category": "Footwear",
        "brand": "Nike",
        "price": 150,
        "specs": "Lightweight mesh upper, Air Max heel unit, rubber outsole, sizes 6-13",
        "description": "White athletic running sneakers with large air cushion heel. Ideal for daily wear and light running.",
        "url": "https://nike.com/p001"
    },
    {
        "id": "P002",
        "name": "Adidas Ultraboost 22 Running Shoes",
        "category": "Footwear",
        "brand": "Adidas",
        "price": 180,
        "specs": "Primeknit upper, Boost midsole, Continental rubber outsole, OrthoLite insole",
        "description": "Black and white performance running shoes with energy-return Boost foam. Best for marathon training.",
        "url": "https://adidas.com/p002"
    },
    {
        "id": "P003",
        "name": "Apple MacBook Pro 14-inch M3",
        "category": "Laptop",
        "brand": "Apple",
        "price": 1999,
        "specs": "Apple M3 chip, 16GB RAM, 512GB SSD, 14.2-inch Liquid Retina XDR display, 18hr battery",
        "description": "Space gray professional laptop with aluminum unibody. Ideal for developers and creative professionals.",
        "url": "https://apple.com/p003"
    },
    {
        "id": "P004",
        "name": "Dell XPS 15 Laptop",
        "category": "Laptop",
        "brand": "Dell",
        "price": 1799,
        "specs": "Intel Core i7-13700H, 16GB DDR5, 512GB NVMe SSD, 15.6-inch OLED 3.5K display, NVIDIA RTX 4060",
        "description": "Silver premium laptop with slim bezels, ideal for creative professionals and power users.",
        "url": "https://dell.com/p004"
    },
    {
        "id": "P005",
        "name": "Sony WH-1000XM5 Headphones",
        "category": "Audio",
        "brand": "Sony",
        "price": 349,
        "specs": "Industry-leading ANC, 30hr battery, LDAC, multipoint connection, speak-to-chat",
        "description": "Black over-ear wireless noise-canceling headphones, best for travel and focus work.",
        "url": "https://sony.com/p005"
    },
    {
        "id": "P006",
        "name": "Samsung Galaxy S24 Ultra",
        "category": "Smartphone",
        "brand": "Samsung",
        "price": 1299,
        "specs": "Snapdragon 8 Gen 3, 200MP camera, 12GB RAM, 256GB, S-Pen, 5000mAh battery",
        "description": "Titanium black flagship Android phone with built-in S-Pen stylus and advanced AI photo features.",
        "url": "https://samsung.com/p006"
    },
    {
        "id": "P007",
        "name": "iPhone 15 Pro Max",
        "category": "Smartphone",
        "brand": "Apple",
        "price": 1199,
        "specs": "A17 Pro chip, 48MP ProRAW camera, titanium frame, 6.7-inch Super Retina XDR, USB-C",
        "description": "Natural titanium premium Apple smartphone with ProRAW photography and action button.",
        "url": "https://apple.com/p007"
    },
    {
        "id": "P008",
        "name": "Levi's 511 Slim Fit Jeans",
        "category": "Clothing",
        "brand": "Levi's",
        "price": 70,
        "specs": "99% cotton 1% elastane, slim fit, mid-rise, available in dark indigo and black",
        "description": "Classic slim-fit dark blue denim jeans for casual and semi-formal wear.",
        "url": "https://levis.com/p008"
    }
]

# Create searchable text for each product (this is what gets embedded)
def build_product_text(p):
    return (
        f"Product: {p['name']}. "
        f"Category: {p['category']}. Brand: {p['brand']}. "
        f"Price: ${p['price']}. "
        f"Description: {p['description']} "
        f"Specs: {p['specs']}"
    )

for p in PRODUCT_CATALOG:
    p["search_text"] = build_product_text(p)

print(f"✅ Loaded {len(PRODUCT_CATALOG)} products into catalog")
print("\nSample product text:")
print(PRODUCT_CATALOG[0]["search_text"])

✅ Loaded 8 products into catalog

Sample product text:
Product: Nike Air Max 270 White Sneakers. Category: Footwear. Brand: Nike. Price: $150. Description: White athletic running sneakers with large air cushion heel. Ideal for daily wear and light running. Specs: Lightweight mesh upper, Air Max heel unit, rubber outsole, sizes 6-13


## Cell 4: Build FAISS Vector Index with HuggingFace Embeddings

In [9]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# ──────────────────────────────────────────────────────────────────────────────
# Initialize HuggingFace embedding model
# all-MiniLM-L6-v2 = 384 dimensions, very fast, good quality
# For higher accuracy: use 'BAAI/bge-large-en-v1.5' (1024-dim)
# ──────────────────────────────────────────────────────────────────────────────
print("Loading HuggingFace embedding model...")
embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},  # change to "cuda" if GPU available
    encode_kwargs={"normalize_embeddings": True}  # normalize for cosine similarity
)

# Convert products to LangChain Document objects
documents = [
    Document(
        page_content=p["search_text"],
        metadata={
            "id":       p["id"],
            "name":     p["name"],
            "price":    p["price"],
            "brand":    p["brand"],
            "category": p["category"],
            "url":      p["url"]
        }
    )
    for p in PRODUCT_CATALOG
]

# Build FAISS vector store
print("Building FAISS index...")
vectorstore = FAISS.from_documents(documents, embedding_model)

# Save index to disk (important for production!)
vectorstore.save_local("faiss_product_index")

print(f"✅ FAISS index built with {len(documents)} products")
print("   Index saved to 'faiss_product_index/'")

# Quick test
test_results = vectorstore.similarity_search("white running shoes Nike", k=2)
print("\nTest search 'white running shoes Nike':")
for r in test_results:
    print(f"  → {r.metadata['name']} (${r.metadata['price']})")

Loading HuggingFace embedding model...
Building FAISS index...
✅ FAISS index built with 8 products
   Index saved to 'faiss_product_index/'

Test search 'white running shoes Nike':
  → Nike Air Max 270 White Sneakers ($150)
  → Adidas Ultraboost 22 Running Shoes ($180)


## Cell 5: Vision Module — Image → Text Description

In [10]:
import base64
import io
import requests
from PIL import Image
from groq import Groq

groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

def image_to_base64(image_source: str) -> tuple:
    """
    Convert image from URL or file path to base64 string.
    Returns (base64_string, media_type)
    """
    if image_source.startswith("http"):
        response = requests.get(image_source)
        image_bytes = response.content
        # Detect format from response headers or default to jpeg
        content_type = response.headers.get("Content-Type", "image/jpeg")
        media_type = content_type.split(";")[0].strip()
    else:
        with open(image_source, "rb") as f:
            image_bytes = f.read()
        # Detect from file extension
        ext = image_source.lower().split(".")[-1]
        media_type = {"jpg": "image/jpeg", "jpeg": "image/jpeg",
                      "png": "image/png", "webp": "image/webp"}.get(ext, "image/jpeg")

    base64_str = base64.b64encode(image_bytes).decode("utf-8")
    return base64_str, media_type


def describe_product_image(image_source: str) -> str:
    """
    Takes an image (file path or URL) and returns a rich text description
    using Llama 4 Scout on Groq — fully multimodal, fast, and free.
    """
    base64_str, media_type = image_to_base64(image_source)

    prompt = """
    You are an expert e-commerce product analyst.
    Analyze this product image and provide a detailed description optimized for search.
    
    Include:
    1. Product category (e.g., sneakers, laptop, headphones, clothing)
    2. Brand (if visible, otherwise say 'Unknown brand')
    3. Color(s) and material
    4. Key visible features and design elements
    5. Apparent use case (e.g., running, gaming, casual wear)
    6. Price range estimate (budget/mid-range/premium)
    
    Format: Write 3-4 sentences as a product description that would help find similar items.
    Be specific and factual. Do not make up specs not visible in the image.
    """

    response = groq_client.chat.completions.create(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:{media_type};base64,{base64_str}"
                        }
                    },
                    {
                        "type": "text",
                        "text": prompt
                    }
                ]
            }
        ],
        max_tokens=512,
        temperature=0.2   # low temp = more factual descriptions
    )

    description = response.choices[0].message.content.strip()
    print(f"📸 Image Description:\n{description}\n")
    return description


# ── TEST ──────────────────────────────────────────────────────────────────────
SAMPLE_IMAGE_URL = "C:/Users/kumar/OneDrive/Desktop/TRY-3/Multimodal-RAG-for-E-commerce-Product-Assistant/images/image.png"

image_description = describe_product_image(SAMPLE_IMAGE_URL)
print("✅ Vision module working with Llama 4 Scout!")

📸 Image Description:
The product in the image is a pair of **sneakers**, specifically from the brand **Puma** as indicated by the logo on the side. The sneakers feature a **black and white color scheme**, with a primarily black upper made of a mesh material and a white midsole and outsole. Key visible features include a thick white midsole, a black lacing system, and a distinctive white stripe design element on the side.

These sneakers appear to be designed for **casual wear or athletic activities** such as running or walking, given their design and construction. Based on the quality of the materials and design, I would estimate that these sneakers fall into the **mid-range** price category, likely priced between $50 to $150.

✅ Vision module working with Llama 4 Scout!


## Cell 6: Retrieval — Find Similar Products from FAISS

In [11]:
def retrieve_similar_products(query_text: str, top_k: int = TOP_K_RESULTS, price_filter: float = None):
    """
    Retrieve top-K similar products from FAISS using semantic search.
    
    Args:
        query_text:   Description of the product (from vision LLM or user text)
        top_k:        Number of results to retrieve
        price_filter: Optional max price filter
    
    Returns:
        List of (Document, score) tuples
    
    💡 TIP: FAISS returns L2 distance; lower = more similar.
           We convert to a 0-1 similarity score for better UX.
    """
    # Semantic search with scores
    results_with_scores = vectorstore.similarity_search_with_score(query_text, k=top_k * 2)
    
    # Optional price filter
    if price_filter:
        results_with_scores = [
            (doc, score) for doc, score in results_with_scores
            if doc.metadata["price"] <= price_filter
        ]
    
    # Take top_k after filter
    results_with_scores = results_with_scores[:top_k]
    
    print(f"🔍 Found {len(results_with_scores)} similar products:")
    for doc, score in results_with_scores:
        similarity = 1 / (1 + score)  # convert L2 distance to similarity
        print(f"  [{similarity:.2%}] {doc.metadata['name']} — ${doc.metadata['price']}")
    
    return results_with_scores


# Test retrieval with our image description
retrieved_products = retrieve_similar_products(
    query_text=image_description,
    top_k=3,
    price_filter=500  # only show products under $500
)

🔍 Found 3 similar products:
  [57.85%] Nike Air Max 270 White Sneakers — $150
  [56.86%] Adidas Ultraboost 22 Running Shoes — $180
  [42.72%] Sony WH-1000XM5 Headphones — $349


## Cell 7: Generation — ChatGroq produces the final answer

In [15]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage

# Initialize ChatGroq
llm = ChatGroq(
    model=GROQ_MODEL,
    temperature=0.3,
    max_tokens=1024,
    groq_api_key=os.environ["GROQ_API_KEY"]
)

def format_products_for_context(results_with_scores):
    """Convert retrieved product docs into a readable context string for LLM."""
    context_parts = []
    for i, (doc, score) in enumerate(results_with_scores, 1):
        m = doc.metadata
        context_parts.append(
            f"Product {i}: {m['name']}\n"
            f"  Brand: {m['brand']} | Category: {m['category']} | Price: ${m['price']}\n"
            f"  Details: {doc.page_content}\n"
            f"  Link: {m['url']}"
        )
    return "\n\n".join(context_parts)


def generate_answer(user_question: str, image_description: str, retrieved_products) -> str:
    product_context = format_products_for_context(retrieved_products)
    
    system_prompt = """You are a helpful AI shopping assistant for an e-commerce platform.
    You help users find products based on images they upload and questions they ask.
    Always be specific, helpful, and honest. If you don't know something, say so.
    Format your response clearly with product names, prices, and key differences."""
    
    user_prompt = f"""The user uploaded a product image. Here's what the image shows:
    
IMAGE DESCRIPTION:
{image_description}

SIMILAR PRODUCTS FOUND IN OUR CATALOG:
{product_context}

USER QUESTION:
{user_question}

Please answer the user's question using the product information above.
If the user asks for alternatives or similar products, list them with prices.
If the user asks about specs, use the product details provided.
End with a helpful buying recommendation."""
    
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt)
    ]
    
    response = llm.invoke(messages)  # ✅ .invoke() instead of llm(messages)
    return response.content


# Test
question = "Show me similar products under $500 and tell me which is the best value?"
answer = generate_answer(question, image_description, retrieved_products)

print("🤖 Assistant Response:")
print("=" * 60)
print(answer)
print("=" * 60)

🤖 Assistant Response:
**Similar Products Under $500:**

Based on the uploaded image of the Puma sneakers, I've found similar products in our catalog that match your criteria. Here are the options:

1. **Nike Air Max 270 White Sneakers**
   - Brand: Nike
   - Category: Footwear
   - Price: $150
   - Description: White athletic running sneakers with large air cushion heel. Ideal for daily wear and light running.
   - Specs: Lightweight mesh upper, Air Max heel unit, rubber outsole, sizes 6-13
   - Link: https://nike.com/p001

2. **Adidas Ultraboost 22 Running Shoes**
   - Brand: Adidas
   - Category: Footwear
   - Price: $180
   - Description: Black and white performance running shoes with energy-return Boost foam. Best for marathon training.
   - Specs: Primeknit upper, Boost midsole, Continental rubber outsole, OrthoLite insole
   - Link: https://adidas.com/p002

**Best Value Recommendation:**

Considering the features and prices, I would recommend the **Nike Air Max 270 White Sneakers

## Cell 8: Full Pipeline — End-to-End Function

In [16]:
import re

def extract_price_filter(question: str):
    """
    Auto-detect price constraints from the user question.
    e.g. 'under $500' → 500, 'below 200 dollars' → 200
    
    💡 TIP: This kind of metadata-aware filtering is what separates
           a demo from a production system.
    """
    patterns = [
        r'under\s*\$?([\d,]+)',
        r'below\s*\$?([\d,]+)',
        r'less\s+than\s*\$?([\d,]+)',
        r'max\s*\$?([\d,]+)',
        r'budget\s*\$?([\d,]+)',
        r'\$?([\d,]+)\s*or\s*less',
    ]
    for pat in patterns:
        match = re.search(pat, question.lower())
        if match:
            return float(match.group(1).replace(',', ''))
    return None


def multimodal_rag_pipeline(
    image_source: str,
    user_question: str,
    top_k: int = TOP_K_RESULTS
) -> dict:
    """
    MAIN PIPELINE: Runs the complete Multimodal RAG flow.
    
    Steps:
    1. Vision LLM   → generate product description from image
    2. Retrieval    → find top-K similar products via FAISS
    3. Generation   → ChatGroq answers the user question
    
    Returns a dict with description, products found, and final answer.
    """
    print("🚀 Starting Multimodal RAG Pipeline")
    print("=" * 50)
    
    # Step 1: Image → Description
    print("\n📸 Step 1: Analyzing image with Vision LLM...")
    description = describe_product_image(image_source)
    
    # Step 2: Description → Similar products
    print("\n🔍 Step 2: Retrieving similar products from FAISS...")
    price_limit = extract_price_filter(user_question)
    if price_limit:
        print(f"   Price filter detected: under ${price_limit}")
    products = retrieve_similar_products(description, top_k=top_k, price_filter=price_limit)
    
    # Step 3: Products + Question → Answer
    print("\n💬 Step 3: Generating answer with ChatGroq...")
    answer = generate_answer(user_question, description, products)
    
    print("\n✅ Pipeline complete!")
    print("\n🤖 FINAL ANSWER:")
    print("=" * 50)
    print(answer)
    print("=" * 50)
    
    return {
        "image_description": description,
        "retrieved_products": [(doc.metadata["name"], doc.metadata["price"]) for doc, _ in products],
        "answer": answer
    }


# ── RUN THE FULL PIPELINE ─────────────────────────────────────────────────────
result = multimodal_rag_pipeline(
    image_source=SAMPLE_IMAGE_URL,
    user_question="What specs does this model have? Also show me similar products under $500."
)

🚀 Starting Multimodal RAG Pipeline

📸 Step 1: Analyzing image with Vision LLM...
📸 Image Description:
The product in the image falls under the category of **sneakers**. The brand appears to be **Puma**, as indicated by the logo on the side of the shoe. The sneaker features a **black and white color scheme**, with a primarily black upper and white midsole and accents; the material appears to be a combination of synthetic materials and mesh. Key visible features include a thick white midsole, a black lacing system, and a distinctive white stripe design element on the side.

This sneaker seems designed for **casual wear or athletic activities** such as running or training. Given the brand and design, it would likely fall into the **mid-range** price category, potentially priced between $80 to $120 USD. 

Here is a product description based on the analysis:

"Black and white Puma sneakers with a thick white midsole and distinctive side stripe. The shoes feature a black upper with synthetic

## Cell 9: Interactive Multi-Turn Chat (Conversation Memory)
> 💡 This is a key differentiator — most demos don't have this!

In [18]:
# ✅ Fix 1: ConversationBufferWindowMemory moved to langchain_community
# ✅ Fix 2: AIMessage moved to langchain_core.messages
# ✅ Fix 3: llm(messages) → llm.invoke(messages)

from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

class MultimodalShoppingChat:
    """
    Multi-turn shopping assistant that remembers conversation context.
    """
    
    def __init__(self):
        self.llm = llm
        self.chat_history = []
        self.current_image_description = None
        self.current_products = None
    
    def load_product_image(self, image_source: str):
        """Load and analyze a new product image."""
        print("📸 Analyzing new product image...")
        self.current_image_description = describe_product_image(image_source)
        self.current_products = retrieve_similar_products(self.current_image_description)
        self.chat_history = []
        print("✅ Image loaded. You can now ask questions!")
        return self.current_image_description
    
    def chat(self, user_message: str) -> str:
        """Send a message and get a response, maintaining conversation history."""
        if not self.current_image_description:
            return "Please load a product image first using load_product_image()."
        
        product_context = format_products_for_context(self.current_products)
        
        messages = [
            SystemMessage(content=f"""You are a helpful shopping assistant.
            
Product being viewed:
{self.current_image_description}

Available similar products:
{product_context}

Answer questions based on this product context. Be concise and helpful."""),
        ]
        
        # Add conversation history (last 3 turns = 6 messages)
        for h in self.chat_history[-6:]:
            messages.append(h)
        
        messages.append(HumanMessage(content=user_message))
        
        response = self.llm.invoke(messages)  # ✅ Fix 3
        answer = response.content
        
        # Save to history
        self.chat_history.append(HumanMessage(content=user_message))
        self.chat_history.append(AIMessage(content=answer))       # ✅ Fix 2
        
        return answer


# Demo
chat = MultimodalShoppingChat()
chat.load_product_image(SAMPLE_IMAGE_URL)

print("\n" + "="*50)
print("Turn 1:")
print(chat.chat("What kind of shoe is this?"))

print("\n" + "="*50)
print("Turn 2:")
print(chat.chat("Which of the similar ones is best for marathon running?"))

print("\n" + "="*50)
print("Turn 3:")
print(chat.chat("And the cheapest option from what you mentioned?"))

📸 Analyzing new product image...
📸 Image Description:
The product in the image falls under the category of **sneakers**. The brand appears to be **Puma**, as indicated by the logo on the side of the shoe. The sneaker features a **black and white color scheme**, with a primarily black upper and white midsole and accents; the material appears to be a combination of synthetic materials and mesh. Key visible features include a thick white midsole, a black lacing system, and a distinctive white stripe design element on the side.

This sneaker seems designed for **casual wear or athletic activities** such as running or training. Given the brand and design, it likely falls into the **mid-range** price category, potentially priced between $80 to $120 USD. A possible product description could be: "Puma black and white athletic sneakers with thick white midsole and synthetic upper, featuring a distinctive side stripe design for casual or running use."

🔍 Found 5 similar products:
  [60.19%] Nike

## Cell 10: 🏆 Bonus Features — What Makes Your Project Stand Out
> Add these to impress interviewers!

In [20]:
!pip install rank-bm25

In [21]:
# ──────────────────────────────────────────────────────────────────────────────
# BONUS 1: Hybrid Search (Semantic + Keyword)
# This significantly improves retrieval quality for product search
# ──────────────────────────────────────────────────────────────────────────────
from rank_bm25 import BM25Okapi  # pip install rank-bm25
import numpy as np

class HybridSearcher:
    """
    Combines semantic (FAISS) + keyword (BM25) search using Reciprocal Rank Fusion.
    
    Why this matters:
    - Semantic search: understands meaning ('running shoes' matches 'athletic footwear')
    - BM25 search: finds exact brand/model names ('Nike Air Max 270')
    - Hybrid: best of both worlds!
    
    💡 TIP: Mention 'RRF hybrid retrieval' in interviews — it shows depth.
    """
    
    def __init__(self, documents, vectorstore, k=60):
        self.docs = documents
        self.vectorstore = vectorstore
        self.k = k  # RRF constant
        
        # Build BM25 index
        tokenized = [doc.page_content.lower().split() for doc in documents]
        self.bm25 = BM25Okapi(tokenized)
    
    def search(self, query: str, top_n: int = 5) -> list:
        """Reciprocal Rank Fusion of semantic + BM25 results."""
        # Semantic results
        semantic_results = self.vectorstore.similarity_search(query, k=top_n * 2)
        semantic_ids = [r.metadata['id'] for r in semantic_results]
        
        # BM25 results
        bm25_scores = self.bm25.get_scores(query.lower().split())
        bm25_ranked = np.argsort(bm25_scores)[::-1][:top_n * 2]
        bm25_ids = [self.docs[i].metadata['id'] for i in bm25_ranked]
        
        # Reciprocal Rank Fusion
        rrf_scores = {}
        for rank, doc_id in enumerate(semantic_ids):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1 / (self.k + rank + 1)
        for rank, doc_id in enumerate(bm25_ids):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1 / (self.k + rank + 1)
        
        # Sort by RRF score
        top_ids = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:top_n]
        
        # Return matching documents
        id_to_doc = {doc.metadata['id']: doc for doc in self.docs}
        return [id_to_doc[id_] for id_ in top_ids if id_ in id_to_doc]


# Initialize hybrid searcher
hybrid_searcher = HybridSearcher(documents, vectorstore)

# Test
hybrid_results = hybrid_searcher.search("Nike white running shoe athletic")
print("🔀 Hybrid Search Results:")
for doc in hybrid_results:
    print(f"  → {doc.metadata['name']} (${doc.metadata['price']})")

🔀 Hybrid Search Results:
  → Nike Air Max 270 White Sneakers ($150)
  → Adidas Ultraboost 22 Running Shoes ($180)
  → Levi's 511 Slim Fit Jeans ($70)
  → Sony WH-1000XM5 Headphones ($349)
  → iPhone 15 Pro Max ($1199)


In [24]:
# ──────────────────────────────────────────────────────────────────────────────
# BONUS 2: Structured Query Extraction
# Parse user intent into structured filters — makes you look like a systems thinker
# ──────────────────────────────────────────────────────────────────────────────
import json
from langchain_core.messages import HumanMessage

def extract_search_intent(user_question: str) -> dict:
    """
    Use LLM to extract structured search intent from natural language.
    
    Example:
    Input:  'Show me white sneakers under $200, preferably Nike or Adidas'
    Output: {category: 'sneakers', max_price: 200, brands: ['Nike', 'Adidas'], color: 'white'}
    """
    prompt = f"""Extract structured search parameters from this user query.

Query: "{user_question}"

Return ONLY a JSON object with these fields (use null if not mentioned):
{{
  "category": "product category (e.g., sneakers, laptop, headphones)",
  "max_price": number or null,
  "min_price": number or null,
  "brands": ["list of brands"] or null,
  "color": "color if mentioned" or null,
  "use_case": "use case if mentioned (e.g., running, gaming, travel)" or null,
  "intent": "find_similar | get_specs | compare | recommend"
}}

Important: Return raw JSON only. No markdown, no explanation, no code fences."""

    response = llm.invoke([HumanMessage(content=prompt)])  # ✅ Fix 1: .invoke()

    try:
        # ✅ Fix 2: robust stripping using regex instead of chained .strip()
        import re
        text = response.content.strip()
        # Remove ```json ... ``` or ``` ... ``` fences if present
        text = re.sub(r'^```(?:json)?\s*', '', text)
        text = re.sub(r'\s*```$', '', text)
        text = text.strip()
        return json.loads(text)
    except json.JSONDecodeError:
        print(f"⚠️ Could not parse JSON. Raw response:\n{response.content}")
        return {"intent": "find_similar"}  # fallback


# Test
intent = extract_search_intent("Show me white sneakers under $200, preferably Nike")
print("🎯 Extracted Intent:")
for key, val in intent.items():
    if val is not None:
        print(f"  {key}: {val}")

🎯 Extracted Intent:
  category: sneakers
  max_price: 200
  brands: ['Nike']
  color: white
  intent: recommend


In [25]:
# ──────────────────────────────────────────────────────────────────────────────
# BONUS 3: Product Comparison Feature
# 'Compare the top 2 results' — very common real-world request
# ──────────────────────────────────────────────────────────────────────────────
def compare_products(product_docs: list, user_criteria: str = None) -> str:
    """
    Generate a structured product comparison using ChatGroq.
    Returns a markdown table comparing the top products.
    """
    products_text = "\n\n".join([
        f"Product {i+1}: {doc.metadata['name']}\n"
        f"Price: ${doc.metadata['price']}\n"
        f"Details: {doc.page_content}"
        for i, (doc, _) in enumerate(product_docs[:3])
    ])
    
    criteria_text = f"Focus on: {user_criteria}" if user_criteria else "Compare all key aspects."
    
    prompt = f"""Compare these products in a clear markdown table format.
{criteria_text}

{products_text}

Create a markdown comparison table with rows for: Price, Key Features, Best For, Pros, Cons.
Then give a 2-sentence recommendation."""
    
    response = llm.invoke([HumanMessage(content=prompt)])  # ✅ only change
    return response.content


# Test
comparison = compare_products(retrieved_products, user_criteria="value for money")
print("📊 Product Comparison:")
print(comparison)

📊 Product Comparison:
### Comparison Table
| Product | Price | Key Features | Best For | Pros | Cons |
| --- | --- | --- | --- | --- | --- |
| Nike Air Max 270 | $150 | Lightweight mesh upper, Air Max heel unit | Daily wear, light running | Affordable, comfortable, stylish | Limited support for heavy running |
| Adidas Ultraboost 22 | $180 | Primeknit upper, Boost midsole, Continental rubber outsole | Marathon training, performance running | High-quality materials, excellent cushioning, durable | Expensive, may not be suitable for casual wear |
| Sony WH-1000XM5 | $349 | Industry-leading ANC, 30hr battery, LDAC, multipoint connection | Travel, focus work, music listening | Excellent noise cancellation, long battery life, high-quality sound | Expensive, may not be suitable for budget-conscious buyers |

Based on the comparison, the Nike Air Max 270 offers the best value for money, providing a great balance of comfort, style, and affordability for daily wear and light running. If you're 

## Cell 11: Save Everything for Production
> Export the index and config so the production app can load them directly

In [26]:
import pickle

# Save FAISS index (already saved above, just confirming)
vectorstore.save_local("faiss_product_index")

# Save product catalog
with open("product_catalog.json", "w") as f:
    json.dump(PRODUCT_CATALOG, f, indent=2)

print("✅ Saved:")
print("  - faiss_product_index/     (vector index for semantic search)")
print("  - product_catalog.json     (product metadata)")
print("\n🎯 Next Steps:")
print("  1. Load these artifacts in your production app")
print("  2. Add real products from an Amazon/Flipkart dataset")
print("  3. Deploy the Streamlit app (see production code)")
print("  4. Add image embeddings for visual similarity (CLIP model)")

✅ Saved:
  - faiss_product_index/     (vector index for semantic search)
  - product_catalog.json     (product metadata)

🎯 Next Steps:
  1. Load these artifacts in your production app
  2. Add real products from an Amazon/Flipkart dataset
  3. Deploy the Streamlit app (see production code)
  4. Add image embeddings for visual similarity (CLIP model)
